# Week 6 - Transformer encoder-decoder sederhana
Marvel Kevin Nathanael | 00000108042

Praktik positional encoding, self-attention kausal, cross-attention, dan prediksi token berikutnya dari TLM 6. Pasangan Portugis-Inggris sintetis ini hanya untuk uji mekanisme.

In [1]:
import numpy as np
import tensorflow as tf

tf.keras.utils.set_random_seed(42)
subjects = [("eu", "i"), ("você", "you"), ("ele", "he"), ("ela", "she")]
verbs = [
    (["vejo", "vê", "vê", "vê"], ["see", "see", "sees", "sees"]),
    (["quero", "quer", "quer", "quer"], ["want", "want", "wants", "wants"]),
    (["bebo", "bebe", "bebe", "bebe"], ["drink", "drink", "drinks", "drinks"]),
]
objects = [("água", "water"), ("café", "coffee"), ("leite", "milk")]

train_pairs, val_pairs, test_pairs = [], [], []
for subject_id, (pt_subject, en_subject) in enumerate(subjects):
    for verb_id, (pt_forms, en_forms) in enumerate(verbs):
        test_object = (subject_id + verb_id) % len(objects)
        other_objects = [i for i in range(len(objects)) if i != test_object]
        val_object = other_objects[0] if (subject_id + verb_id) % 2 == 0 else -1
        for object_id, (pt_object, en_object) in enumerate(objects):
            pair = (f"{pt_subject} {pt_forms[subject_id]} {pt_object}",
                    f"{en_subject} {en_forms[subject_id]} {en_object}")
            if object_id == test_object:
                test_pairs.append(pair)
            elif object_id == val_object:
                val_pairs.append(pair)
            else:
                train_pairs.append(pair)

src_vocab = ["<pad>", "<unk>"] + sorted({t for s, _ in train_pairs for t in s.split()})
tgt_vocab = ["<pad>", "<unk>", "<start>", "<end>"] + sorted(
    {t for _, s in train_pairs for t in s.split()})
src_to_id = {t: i for i, t in enumerate(src_vocab)}
tgt_to_id = {t: i for i, t in enumerate(tgt_vocab)}
id_to_tgt = dict(enumerate(tgt_vocab))
def arrays(data):
    src = np.array([[src_to_id.get(t, 1) for t in a.split()]
                    for a, _ in data], dtype="int32")
    target = np.array([[tgt_to_id.get(t, 1) for t in b.split()] + [tgt_to_id["<end>"]]
                       for _, b in data], dtype="int32")
    dec_in = np.concatenate([
        np.full((len(data), 1), tgt_to_id["<start>"], dtype="int32"),
        target[:, :-1]], axis=1)
    return (src, dec_in), target
(x_train, y_train), (x_val, y_val), (x_test, y_test) = map(
    arrays, [train_pairs, val_pairs, test_pairs])
print("Train/val/test:", len(train_pairs), len(val_pairs), len(test_pairs))
print("Contoh:", train_pairs[0])


Train/val/test: 18 6 12
Contoh: ('eu vejo leite', 'i see milk')


In [2]:
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, d_model, max_len=16):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model)
        pos = np.arange(max_len)[:, None]
        depth = np.arange(d_model)[None, :]
        angles = pos / np.power(10000.0, (2 * (depth // 2)) / d_model)
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])
        self.position = tf.constant(angles[None, ...], dtype=tf.float32)
        self.scale = np.sqrt(d_model)

    def call(self, tokens):
        x = self.embedding(tokens) * self.scale
        return x + self.position[:, :tf.shape(tokens)[1], :]

D_MODEL = 64
src = tf.keras.Input(shape=(None,), dtype=tf.int32, name="portuguese")
dec = tf.keras.Input(shape=(None,), dtype=tf.int32, name="english_prefix")
x = PositionalEmbedding(len(src_vocab), D_MODEL)(src)
attn = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
x = tf.keras.layers.LayerNormalization()(x + attn)
ffn = tf.keras.layers.Dense(D_MODEL, activation="relu")(x)
x = tf.keras.layers.LayerNormalization()(x + ffn)

y = PositionalEmbedding(len(tgt_vocab), D_MODEL)(dec)
self_attn = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=16)(
    y, y, use_causal_mask=True)
y = tf.keras.layers.LayerNormalization()(y + self_attn)
cross = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=16)(y, x)
y = tf.keras.layers.LayerNormalization()(y + cross)
ffn = tf.keras.layers.Dense(D_MODEL, activation="relu")(y)
y = tf.keras.layers.LayerNormalization()(y + ffn)
logits = tf.keras.layers.Dense(len(tgt_vocab))(y)
model = tf.keras.Model([src, dec], logits)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["sparse_categorical_accuracy"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ portuguese          │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 64)  │        960 │ portuguese[0][0]  │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 64)  │     16,640 │ positional_embed… │
│ (MultiHeadAttentio… │                   │            │ positional_embed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ english_prefix      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, None, 64)  │          0 │ positional_embed… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 64)  │      1,088 │ english_prefix[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, None, 64)  │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 64)  │     16,640 │ positional_embed… │
│ (MultiHeadAttentio… │                   │            │ positional_embed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 64)  │      4,160 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, None, 64)  │          0 │ positional_embed… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, None, 64)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 64)  │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 64)  │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 64)  │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, None, 64)  │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 64)  │        128 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 64)  │      4,160 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

Total params: 62,033 (242.32 KB)

Trainable params: 62,033 (242.32 KB)

Non-trainable params: 0 (0.00 B)

In [3]:
history = model.fit(
    x_train, y_train, validation_data=(x_val, y_val),
    epochs=150, batch_size=8, verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=15, restore_best_weights=True)])
print("Epoch terpakai:", len(history.history["loss"]))

def translate(sentence):
    source = np.array([[src_to_id.get(t, 1) for t in sentence.split()]], dtype="int32")
    prefix = [tgt_to_id["<start>"]]
    for _ in range(4):
        pred = model.predict([source, np.array([prefix], dtype="int32")], verbose=0)
        token = int(np.argmax(pred[0, -1]))
        if token == tgt_to_id["<end>"]:
            break
        prefix.append(token)
    return " ".join(id_to_tgt[i] for i in prefix[1:])

for src_text, expected in test_pairs:
    print(f"{src_text:20s} -> {translate(src_text):20s} | target: {expected}")
exact = np.mean([translate(a) == b for a, b in test_pairs])
print(f"Exact-match test: {exact:.3f}")

Epoch terpakai: 150


eu vejo água         -> i see water          | target: i see water


eu quero café        -> i want coffee        | target: i want coffee


eu bebo leite        -> i drink milk         | target: i drink milk


você vê café         -> you see coffee       | target: you see coffee


...

ele bebe café        -> he drinks coffee     | target: he drinks coffee


ela vê água          -> she sees water       | target: she sees water


ela quer café        -> she wants coffee     | target: she wants coffee


ela bebe leite       -> she drinks milk      | target: she drinks milk


Exact-match test: 1.000


**Catatan:** Causal mask mencegah decoder melihat token berikutnya saat training. Test berisi kombinasi subjek, verba, dan objek yang berbeda dari train, tetapi kosakatanya sama. Exact match pada data sintetis ini belum mewakili terjemahan nyata.